# CC-DQML Sprint 1 Walkthrough

This notebook mirrors the Sprint 1 command-line run step by step so the scripts can be inspected interactively.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(path for path in candidates if (path / "src" / "cc_dqml").exists())
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".cache" / "matplotlib"))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

In [2]:
from cc_dqml.config import load_config
from cc_dqml.data import generate_synthetic_dataset
from cc_dqml.circuits import initialize_parameters, make_cc_dqml_qnode, parameters_to_dict
from cc_dqml.train import run_experiment

In [3]:
config = load_config(repo_root / "config" / "cc_dqml.yaml")
config

ExperimentConfig(experiment=ExperimentSettings(model='cc_dqml', output_dir=PosixPath('results/sprint1_cc_dqml'), dataset_seed=0, init_seed=0), data=DataSettings(n_features=8, n_samples=2048, n_clusters=32, train_size=1536, validation_size=512, sphere_radius=0.7853981633974483), model=ModelSettings(qpus=2, qubits_per_qpu=4, convolutional_sub_layers=3, communication='classical', interpret_weights_initial=(1.0, -1.0, -1.0, 1.0)), training=TrainingSettings(optimizer='adam', learning_rate=0.05, batch_size=32, iterations=10))

In [5]:
dataset = generate_synthetic_dataset(config.data, config.experiment.dataset_seed)
dataset.x_train.shape, dataset.x_val.shape, dataset.max_abs_pearson
print(f"Dataset: {dataset.x_train.shape[0]} training samples, {dataset.x_val.shape[0]} validation samples")

Dataset: 1536 training samples, 512 validation samples


In [6]:
params = initialize_parameters(
    config.model.convolutional_sub_layers,
    config.model.interpret_weights_initial,
    config.experiment.init_seed,
)
{name: value.shape for name, value in parameters_to_dict(params).items()}

{'conv': (3, 2, 4),
 'pool': (2, 3, 4),
 'feedforward': (4, 4),
 'interpret': (4,)}

In [7]:
named_params = parameters_to_dict(params)
circuit = make_cc_dqml_qnode()
probs = circuit(dataset.x_train[0], named_params["conv"], named_params["pool"], named_params["feedforward"])
probs

tensor([0.27084983, 0.36265694, 0.19077642, 0.17571681], requires_grad=True)

In [8]:
summary = run_experiment(config)
summary

iteration=1 train_loss=1.0241 val_acc=0.6133
iteration=5 train_loss=0.9440 val_acc=0.6914
iteration=10 train_loss=0.9634 val_acc=0.6113


{'model': 'cc_dqml',
 'communication': 'classical',
 'convolutional_sub_layers': 3.0,
 'dataset_seed': 0.0,
 'init_seed': 0.0,
 'max_abs_pearson': 0.29602050772864086,
 'train_loss': 0.9217640963561093,
 'train_accuracy': 0.6080729166666666,
 'validation_loss': 0.9181538246418364,
 'validation_accuracy': 0.611328125}